<a href="https://colab.research.google.com/github/DylanAttlesey/NFIP-Flood-Severity/blob/main/NFIPDataWrangling_SingResSeverity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Setup & Environment
Mounts Google Drive, sets up Git, and loads the basic county classification map.

In [1]:
import pandas as pd
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Git Configuration
!git config --global user.email "attleseyd@gmail.com"
!git config --global user.name "DylanAttlesey"

# 3. Load the county classification dataset
csv_path = '/content/drive/MyDrive/Programming/nfip/county_classification.csv'

if os.path.exists(csv_path):
    county_classes_df = pd.read_csv(csv_path, dtype={'FIPS': str})
    print(f"Successfully loaded {len(county_classes_df)} records from {csv_path}")
    display(county_classes_df.head())
else:
    print(f"Error: File not found at {csv_path}")

Mounted at /content/drive
Successfully loaded 3235 records from /content/drive/MyDrive/Programming/nfip/county_classification.csv


,FIPS,STATE,COUNTY_NAME,SHORELINE_FLAG,WATERSHED_FLAG
0,01001,AL,Autauga County,0,0
1,01003,AL,Baldwin County,0,0
2,01005,AL,Barbour County,0,0
3,01007,AL,Bibb County,0,0
4,01009,AL,Blount County,0,0


### 2. NFIP Claims Processing
Loads the primary NFIP dataset, filters for relevant dates and building types, and engineers core features (imputing missing values, standardizing categoricals).

In [2]:
import datetime
import pandas as pd
import os

# 1. Load raw data filtered for originalNBDate >= 2011
file_path = '/content/drive/MyDrive/Programming/nfip/FimaNfipClaimsV2.parquet'

if not os.path.exists(file_path):
    print(f"Error: File not found at {file_path}. Please check the path.")
else:
    # Define the start date
    start_date = datetime.date(2011, 1, 1)

    # Define columns to load (removed exclusions, ensured lowestAdjacentGrade is present)
    columns_to_load = [
        'primaryResidenceIndicator', 'rentalPropertyIndicator', 'basementEnclosureCrawlspaceType',
        'crsClassificationCode', 'elevatedBuildingIndicator', 'elevationDifference', 'baseFloodElevation',
        'ratedFloodZone', 'lowestAdjacentGrade', 'lowestFloorElevation',
        'numberOfFloorsInTheInsuredBuilding', 'obstructionType', 'occupancyType', 'originalConstructionDate',
        'originalNBDate', 'postFIRMConstructionIndicator',
        'floodproofedIndicator', 'floodZoneCurrent',
        'buildingDescriptionCode', 'countyCode', 'censusTract', 'censusBlockGroupFips', 'amountPaidOnBuildingClaim',
         'buildingDamageAmount',
        'netBuildingPaymentAmount', 'buildingPropertyValue', 'buildingReplacementCost', 'asOfDate', 'dateOfLoss'
    ]

    # Load with filters to minimize memory usage
    nfip_df = pd.read_parquet(
        file_path,
        columns=columns_to_load,
        filters=[
            ('originalNBDate', '>=', start_date)
        ]
    )

    # Filter specific occupancy types: {1, 11, 14}
    nfip_df = nfip_df[nfip_df['occupancyType'].isin([1, 11, 14, '1', '11', '14'])]

    # Filter buildingDescriptionCode: {1, 18, 19} (accounting for strings/ints like '01')
    nfip_df = nfip_df[nfip_df['buildingDescriptionCode'].isin([1, 18, 19, '01', '1', '18', '19'])]

    # Extract underwriting year from originalNBDate, coercing invalid dates to NaT
    nfip_df['originalNBDate'] = pd.to_datetime(nfip_df['originalNBDate'], errors='coerce')
    nfip_df['uw_year'] = nfip_df['originalNBDate'].dt.year

    # Extract yearOfLoss from dateOfLoss
    nfip_df['dateOfLoss'] = pd.to_datetime(nfip_df['dateOfLoss'], errors='coerce')
    nfip_df['yearOfLoss'] = nfip_df['dateOfLoss'].dt.year

    print("--- Applying Feature Engineering & Cleaning ---")

    # Drop null buildingDamageAmount
    nfip_df = nfip_df.dropna(subset=['buildingDamageAmount'])

    # 1. Mobile Home Engineering & Consolidation
    nfip_df['is_mobile_home'] = (
        nfip_df['occupancyType'].isin([14, '14', 14.0]) |
        nfip_df['buildingDescriptionCode'].isin([18, 19, '18', '19', 18.0, 19.0])
    ).astype(int)
    nfip_df.loc[nfip_df['is_mobile_home'] == 1, 'numberOfFloorsInTheInsuredBuilding'] = 1.0
    nfip_df = nfip_df.drop(columns=['occupancyType', 'buildingDescriptionCode'])

    # 2. Flood Zone Binning
    def map_flood_zone(zone):
        if pd.isna(zone): return 'Unknown'
        z = str(zone).upper().strip()
        if z.startswith('V'): return 'V Zones'
        if z.startswith('A'): return 'A Zones'
        if z.startswith('D'): return 'Zone D'
        if z == 'B': return 'Zone B & X (shaded)'
        if z in ['C', 'X']: return 'Zone C & X (unshaded)'
        return 'Unknown'
    nfip_df['floodZoneCurrent'] = nfip_df['floodZoneCurrent'].apply(map_flood_zone)
    if 'ratedFloodZone' in nfip_df.columns:
        nfip_df = nfip_df.drop(columns=['ratedFloodZone'])

    # 3. Categorical Standardization
    def map_obstruction(obs):
        if pd.isna(obs): return 'Unknown'
        val = str(obs).strip()
        if val in ['10', '10.0', '1']: return 'Free of Obstruction'
        if val == 'Unknown': return 'Unknown'
        return 'Obstructed'

    nfip_df['obstructionType'] = nfip_df['obstructionType'].fillna('Unknown').apply(map_obstruction)

    def map_basement(val):
        if pd.isna(val): return 'Unknown'
        v = str(val).strip().replace('.0', '')
        mapping = {
            '0': 'None',
            '1': 'Finished Basement',
            '2': 'Unfinished Basement',
            '3': 'Crawlspace',
            '4': 'Subgrade Crawlspace'
        }
        return mapping.get(v, 'Unknown')

    nfip_df['basementEnclosureCrawlspaceType'] = nfip_df['basementEnclosureCrawlspaceType'].apply(map_basement)

    # 4. Indicator Casting
    indicator_cols = ['primaryResidenceIndicator', 'rentalPropertyIndicator', 'elevatedBuildingIndicator']
    for col in indicator_cols:
        nfip_df[col] = nfip_df[col].map({
            True: 'True', False: 'False',
            'True': 'True', 'False': 'False',
            1: 'True', 0: 'False',
            1.0: 'True', 0.0: 'False'
        }).fillna('Unknown')

    # 5. Floor Count Imputation
    nfip_df['numberOfFloorsInTheInsuredBuilding'] = nfip_df['numberOfFloorsInTheInsuredBuilding'].fillna(1.0)

    # 6. Administrative Exclusions
    if 'crsClassificationCode' in nfip_df.columns:
        nfip_df = nfip_df.drop(columns=['crsClassificationCode'])

    # 7. Collinearity Prevention & Sparse Numeric Elevation Handling
    if 'lowestFloorElevation' in nfip_df.columns:
        nfip_df = nfip_df.drop(columns=['lowestFloorElevation'])

    # Lowest Adjacent Grade
    nfip_df['is_lag_missing'] = nfip_df['lowestAdjacentGrade'].isna().astype(int)
    nfip_df['lowestAdjacentGrade'] = nfip_df['lowestAdjacentGrade'].fillna(0)

    # Elevation Difference
    nfip_df['is_elev_diff_missing'] = nfip_df['elevationDifference'].isna().astype(int)
    nfip_df['elevationDifference'] = nfip_df['elevationDifference'].fillna(0)

    # Base Flood Elevation
    nfip_df['is_bfe_missing'] = nfip_df['baseFloodElevation'].isna().astype(int)
    nfip_df['baseFloodElevation'] = nfip_df['baseFloodElevation'].fillna(0)

    print(f"Loaded and engineered {len(nfip_df):,} records from 2011 onwards after applying all filters and transformations.")
    display(nfip_df.head())


--- Applying Feature Engineering & Cleaning ---
Loaded and engineered 292,109 records from 2011 onwards after applying all filters and transformations.


,primaryResidenceIndicator,rentalPropertyIndicator,basementEnclosureCrawlspaceType,elevatedBuildingIndicator,elevationDifference,baseFloodElevation,lowestAdjacentGrade,numberOfFloorsInTheInsuredBuilding,obstructionType,originalConstructionDate,...,buildingPropertyValue,buildingReplacementCost,asOfDate,dateOfLoss,uw_year,yearOfLoss,is_mobile_home,is_lag_missing,is_elev_diff_missing,is_bfe_missing
2,False,False,Unknown,False,0,0,0,2.0,Unknown,1950-07-01,...,205878.0,242209.0,2026-06-01 00:00:00+00:00,2014-08-05,2014,2014,0,1,1,1
3,True,False,Unknown,False,0,0,0,1.0,Unknown,1974-01-01,...,69269.0,115447.0,2026-06-01 00:00:00+00:00,2011-06-18,2011,2011,0,1,1,1
5,True,False,Unknown,False,0,0,0,1.0,Unknown,1978-01-01,...,189891.0,358644.0,2026-06-01 00:00:00+00:00,2017-09-10,2013,2017,0,1,1,1
6,True,False,None,False,0,0,0,1.0,Unknown,1983-06-15,...,228976.0,285605.0,2026-06-01 00:00:00+00:00,2022-09-28,2017,2022,0,1,1,1
9,True,False,Finished Basement,False,0,0,0,3.0,Unknown,1975-01-01,...,134943.0,305324.0,2026-06-01 00:00:00+00:00,2013-09-12,2011,2013,0,1,1,1


### 3. Sparsity Analysis
A helper function to calculate and display the percentage of missing data across all columns after our initial cleaning.

In [3]:
def check_sparsity(df):
    """
    Calculates the sparsity (percentage of missing values) for all columns in the DataFrame.
    """
    total_rows = len(df)
    missing_counts = df.isna().sum()
    sparsity_pct = (missing_counts / total_rows) * 100

    # Combine into a DataFrame for a clean report
    sparsity_report = pd.DataFrame({
        'Missing Values': missing_counts,
        'Sparsity (%)': sparsity_pct
    })

    # Sort by highest sparsity first
    sparsity_report = sparsity_report.sort_values(by='Sparsity (%)', ascending=False)

    print(f"Sparsity Report (Total records: {total_rows:,}):")
    display(sparsity_report)

    return sparsity_report

# Example usage:
check_sparsity(nfip_df)


Sparsity Report (Total records: 292,109):


,Missing Values,Sparsity (%)
amountPaidOnBuildingClaim,15844,5.424003
censusTract,968,0.331383
censusBlockGroupFips,968,0.331383
countyCode,573,0.196160
originalConstructionDate,3,0.001027
buildingReplacementCost,2,0.000685
buildingPropertyValue,2,0.000685
lowestAdjacentGrade,0,0.000000
baseFloodElevation,0,0.000000
primaryResidenceIndicator,0,0.000000


,Missing Values,Sparsity (%)
amountPaidOnBuildingClaim,15844,5.424003
censusTract,968,0.331383
censusBlockGroupFips,968,0.331383
countyCode,573,0.196160
originalConstructionDate,3,0.001027
buildingReplacementCost,2,0.000685
buildingPropertyValue,2,0.000685
lowestAdjacentGrade,0,0.000000
baseFloodElevation,0,0.000000
primaryResidenceIndicator,0,0.000000


In [4]:
print(f"Records before dropping: {len(nfip_df):,}")

cols_to_check = ['originalConstructionDate', 'buildingReplacementCost', 'buildingPropertyValue']
nfip_df = nfip_df.dropna(subset=cols_to_check)

print(f"Records after dropping missing values: {len(nfip_df):,}")

Records before dropping: 292,109
Records after dropping missing values: 292,104


### 4. NHGIS Data Compilation
Reads through multiple years of raw NHGIS housing data, standardizing columns and FIPS codes to build a unified median home value dataset.

In [5]:
import os
import glob
import re
import pandas as pd

def build_med_price_df(nhgis_dir):
    csv_files = glob.glob(os.path.join(nhgis_dir, '*_blck_grp.csv'))
    df_list = []

    for file in csv_files:
        # Extract the end year from the filename (e.g., '20095' -> 2009)
        match = re.search(r'_(\d{4})5_blck_grp\.csv', os.path.basename(file))
        if not match:
            continue

        end_year = int(match.group(1))
        uw_year = end_year + 2  # 2005-2009 gets uw_year 2011

        # Add low_memory=False to fix DtypeWarnings
        df = pd.read_csv(file, encoding='latin1', low_memory=False)

        # Dynamically identify the median value column using the descriptive row
        desc_row = df.iloc[0]
        val_col = None
        for col in df.columns:
            desc = str(desc_row[col]).lower()
            # We want the estimate for Median Home Value, not the Margin of Error
            if 'median' in desc and 'error' not in desc:
                val_col = col
                break

        # Fallback if description matching fails
        if not val_col:
            ignore_cols = {'GISJOIN', 'YEAR', 'REGIONA', 'DIVISIONA', 'STATEA', 'COUNTYA',
                           'COUSUBA', 'PLACEA', 'TRACTA', 'BLKGRPA', 'CONCITA', 'PRMA',
                           'ROOTA', 'SDELMA', 'SDSECA', 'SDUNIA', 'URBA', 'NAME', 'STATE', 'COUNTY'}
            est_cols = [c for c in df.columns if c.endswith('E') and len(c) in (7, 8) and c not in ignore_cols]
            if est_cols:
                val_col = est_cols[-1]
            else:
                val_cols = [c for c in df.columns if c not in ignore_cols and not c.endswith('M')]
                if not val_cols:
                    continue
                val_col = val_cols[0]

        # Drop the secondary descriptive header row included in NHGIS extracts
        df = df.iloc[1:].reset_index(drop=True)

        temp_df = df[['GISJOIN', val_col]].copy()
        temp_df.rename(columns={val_col: 'median_home_value'}, inplace=True)
        temp_df['uw_year'] = uw_year

        # 1. Drop the 'G'
        temp_df['clean_fips'] = temp_df['GISJOIN'].str.replace('G', '')

        # 2. Slice out the extra IPUMS zeros
        temp_df['censusBlockGroupFips'] = (
            temp_df['clean_fips'].str[0:2] +   # State
            temp_df['clean_fips'].str[3:6] +   # County
            temp_df['clean_fips'].str[7:]      # Tract & Block Group
        )

        # Select final columns
        df_list.append(temp_df[['uw_year', 'censusBlockGroupFips', 'median_home_value']])

    compiled_df = pd.concat(df_list, ignore_index=True)
    # Coerce to numeric immediately to turn '.' suppression strings into NaN
    compiled_df['median_home_value'] = pd.to_numeric(compiled_df['median_home_value'], errors='coerce')
    return compiled_df

# Execute compilation
nhgis_dir = '/content/drive/MyDrive/Programming/nfip/nhgis'
med_price_df = build_med_price_df(nhgis_dir)

print(f"Successfully compiled med_price_df with {len(med_price_df):,} records.")
display(med_price_df.head())


Successfully compiled med_price_df with 3,626,204 records.


,uw_year,censusBlockGroupFips,median_home_value
0,2024,010010201001,176600.0
1,2024,010010201002,154000.0
2,2024,010010202001,137500.0
3,2024,010010202002,125300.0
4,2024,010010203001,137900.0


In [6]:
print("--- Non-Numeric Artifact Analysis in NHGIS Data ---")

# Count exact matches of '.'
dot_count = (med_price_df['median_home_value'] == '.').sum()
total_records = len(med_price_df)
print(f"Number of '.' values: {dot_count:,} out of {total_records:,} ({dot_count/total_records*100:.2f}%)")

# Find any other non-numeric strings that might exist
non_numeric_mask = pd.to_numeric(med_price_df['median_home_value'], errors='coerce').isna()
non_numeric_vals = med_price_df.loc[non_numeric_mask, 'median_home_value'].value_counts(dropna=False)

print("\nAll non-numeric/null values found in raw compiled data:")
print(non_numeric_vals)

--- Non-Numeric Artifact Analysis in NHGIS Data ---
Number of '.' values: 0 out of 3,626,204 (0.00%)

All non-numeric/null values found in raw compiled data:
median_home_value
NaN    183709
Name: count, dtype: int64


### 5. Temporal Imputation (Grid Approach)
Fills in gaps in the housing data by carrying forward previous years' prices, then merges this economic data into our main NFIP claims dataset.

In [7]:
print("--- Imputing Missing Values in NHGIS Data (Grid Approach) ---")

# 1. Create a complete grid of all FIPS codes and all years
all_fips = med_price_df['censusBlockGroupFips'].unique()
all_years = range(med_price_df['uw_year'].min(), med_price_df['uw_year'].max() + 1)
full_index = pd.MultiIndex.from_product([all_fips, all_years], names=['censusBlockGroupFips', 'uw_year'])

# 2. Reindex the dataframe to ensure every FIPS-Year combination exists as a row
med_price_grid = med_price_df.set_index(['censusBlockGroupFips', 'uw_year']).reindex(full_index).reset_index()

# 3. Ensure chronological order for proper filling
med_price_grid = med_price_grid.sort_values(['censusBlockGroupFips', 'uw_year'])

# 4. Forward Fill: For missing medians at time t, use t-d (smallest possible d)
med_price_grid['median_home_value'] = med_price_grid.groupby('censusBlockGroupFips')['median_home_value'].ffill()

# 5. Backfill: Specifically for 2011, use the 2012 median if it exists
med_price_grid['median_home_value'] = med_price_grid.groupby('censusBlockGroupFips')['median_home_value'].bfill(limit=1)

# Determine the maximum year available in the median price dataset
max_med_year = med_price_grid['uw_year'].max()

# Create a capped underwriting year column for the merge
nfip_df['merge_uw_year'] = nfip_df['uw_year'].clip(upper=max_med_year)

# Merge the NFIP claims with the expanded NHGIS median home values grid
merged_df = pd.merge(nfip_df, med_price_grid, left_on=['censusBlockGroupFips', 'merge_uw_year'], right_on=['censusBlockGroupFips', 'uw_year'], how='left')

# Clean up the extra columns created by the merge keys
merged_df = merged_df.drop(columns=['merge_uw_year', 'uw_year_y']).rename(columns={'uw_year_x': 'uw_year'})

print(f"Successfully merged DataFrames. Resulting records: {len(merged_df):,}")
display(merged_df.head())


--- Imputing Missing Values in NHGIS Data (Grid Approach) ---
Successfully merged DataFrames. Resulting records: 292,104


,primaryResidenceIndicator,rentalPropertyIndicator,basementEnclosureCrawlspaceType,elevatedBuildingIndicator,elevationDifference,baseFloodElevation,lowestAdjacentGrade,numberOfFloorsInTheInsuredBuilding,obstructionType,originalConstructionDate,...,buildingReplacementCost,asOfDate,dateOfLoss,uw_year,yearOfLoss,is_mobile_home,is_lag_missing,is_elev_diff_missing,is_bfe_missing,median_home_value
0,False,False,Unknown,False,0,0,0,2.0,Unknown,1950-07-01,...,242209.0,2026-06-01 00:00:00+00:00,2014-08-05,2014,2014,0,1,1,1,1000001.0
1,True,False,Unknown,False,0,0,0,1.0,Unknown,1974-01-01,...,115447.0,2026-06-01 00:00:00+00:00,2011-06-18,2011,2011,0,1,1,1,79400.0
2,True,False,Unknown,False,0,0,0,1.0,Unknown,1978-01-01,...,358644.0,2026-06-01 00:00:00+00:00,2017-09-10,2013,2017,0,1,1,1,381700.0
3,True,False,None,False,0,0,0,1.0,Unknown,1983-06-15,...,285605.0,2026-06-01 00:00:00+00:00,2022-09-28,2017,2022,0,1,1,1,401200.0
4,True,False,Finished Basement,False,0,0,0,3.0,Unknown,1975-01-01,...,305324.0,2026-06-01 00:00:00+00:00,2013-09-12,2011,2013,0,1,1,1,409600.0


### 6. Spatial Imputation
Catches any remaining missing home values in highly sparse areas by calculating and applying the median price of the broader surrounding census tract or county.

In [8]:
import numpy as np
import pandas as pd

print("--- Spatial Imputation for Missing Median Home Values ---")

# 0. Ensure median_home_value is numeric to avoid aggregation errors
med_price_grid['median_home_value'] = pd.to_numeric(med_price_grid['median_home_value'], errors='coerce')

# 1. Derive tract and county FIPS in the grid to aggregate spatial medians
med_price_grid['tract_fips'] = med_price_grid['censusBlockGroupFips'].str[:11]
med_price_grid['county_fips'] = med_price_grid['censusBlockGroupFips'].str[:5]

# 2. Calculate medians at tract and county levels per year
tract_medians = med_price_grid.groupby(['tract_fips', 'uw_year'])['median_home_value'].median().reset_index()
county_medians = med_price_grid.groupby(['county_fips', 'uw_year'])['median_home_value'].median().reset_index()

# 3. Add spatial keys and capped year to merged_df for merging
merged_df['tract_fips'] = merged_df['censusBlockGroupFips'].astype(str).str[:11]
merged_df['county_fips'] = merged_df['censusBlockGroupFips'].astype(str).str[:5]
merged_df['merge_uw_year'] = merged_df['uw_year'].clip(upper=med_price_grid['uw_year'].max())

# 4. Initialize imputation tracker as object to avoid warnings when assigning strings
merged_df['imputation'] = np.nan
merged_df['imputation'] = merged_df['imputation'].astype(object)

# 5. Impute Tract Level
merged_df = merged_df.merge(tract_medians, left_on=['tract_fips', 'merge_uw_year'], right_on=['tract_fips', 'uw_year'], how='left', suffixes=('', '_tract'))
mask_tract = merged_df['median_home_value'].isna() & merged_df['median_home_value_tract'].notna()
merged_df.loc[mask_tract, 'median_home_value'] = merged_df.loc[mask_tract, 'median_home_value_tract']
merged_df.loc[mask_tract, 'imputation'] = 'tract'

# 6. Impute County Level
merged_df = merged_df.merge(county_medians, left_on=['county_fips', 'merge_uw_year'], right_on=['county_fips', 'uw_year'], how='left', suffixes=('', '_county'))
mask_county = merged_df['median_home_value'].isna() & merged_df['median_home_value_county'].notna()
merged_df.loc[mask_county, 'median_home_value'] = merged_df.loc[mask_county, 'median_home_value_county']
merged_df.loc[mask_county, 'imputation'] = 'county'

# 7. Clean up merge artifacts
cols_to_drop = ['tract_fips', 'county_fips', 'merge_uw_year',
                'uw_year_tract', 'median_home_value_tract',
                'uw_year_county', 'median_home_value_county']
merged_df = merged_df.drop(columns=cols_to_drop)

print("Imputation breakdown:")
print(merged_df['imputation'].value_counts(dropna=False))
print("\n")

# 8. Check resulting sparsity
check_sparsity(merged_df)


--- Spatial Imputation for Missing Median Home Values ---
Imputation breakdown:
imputation
NaN       254939
county     32473
tract       4692
Name: count, dtype: int64


Sparsity Report (Total records: 292,104):


,Missing Values,Sparsity (%)
imputation,254939,87.276792
amountPaidOnBuildingClaim,15844,5.424096
median_home_value,1020,0.349191
censusTract,968,0.331389
censusBlockGroupFips,968,0.331389
countyCode,573,0.196163
primaryResidenceIndicator,0,0.000000
rentalPropertyIndicator,0,0.000000
numberOfFloorsInTheInsuredBuilding,0,0.000000
lowestAdjacentGrade,0,0.000000


,Missing Values,Sparsity (%)
imputation,254939,87.276792
amountPaidOnBuildingClaim,15844,5.424096
median_home_value,1020,0.349191
censusTract,968,0.331389
censusBlockGroupFips,968,0.331389
countyCode,573,0.196163
primaryResidenceIndicator,0,0.000000
rentalPropertyIndicator,0,0.000000
numberOfFloorsInTheInsuredBuilding,0,0.000000
lowestAdjacentGrade,0,0.000000


### 7. Final Cleanup
Drop records where `median_home_value` could not be imputed.

In [9]:
print(f"Records before dropping missing median_home_value: {len(merged_df):,}")

# Drop records that are still missing median_home_value
merged_df = merged_df.dropna(subset=['median_home_value'])

print(f"Records after dropping: {len(merged_df):,}")


Records before dropping missing median_home_value: 292,104
Records after dropping: 291,084


### 8. Merge NOAA County Classification Data
Joins the county-level NOAA climate/classification data using a standardized 5-digit FIPS code.

In [10]:
print("--- Merging NOAA County Classification Data ---")

# 1. Standardize county FIPS to 5 digits
# Try countyCode first, handling any float '.0' artifacts and nulls
c_code = merged_df['countyCode'].astype(str).str.replace(r'\.0$', '', regex=True)
c_code = c_code.where(c_code != 'nan').str.zfill(5)

# Use the first 5 digits of censusBlockGroupFips as a fallback
bg_code = merged_df['censusBlockGroupFips'].astype(str).str[:5]
bg_code = bg_code.where(bg_code != 'nan')

merged_df['merge_fips'] = c_code.fillna(bg_code)

# 2. Perform the left merge with the county_classes_df loaded in Section 1
merged_df = merged_df.merge(county_classes_df, left_on='merge_fips', right_on='FIPS', how='left')

# 3. Clean up the temporary merge key
merged_df = merged_df.drop(columns=['merge_fips'])

print(f"Successfully merged NOAA county data.")
print(f"Final dataset shape: {merged_df.shape[0]:,} records, {merged_df.shape[1]} columns")
display(merged_df.head())

--- Merging NOAA County Classification Data ---
Successfully merged NOAA county data.
Final dataset shape: 291,084 records, 37 columns


,primaryResidenceIndicator,rentalPropertyIndicator,basementEnclosureCrawlspaceType,elevatedBuildingIndicator,elevationDifference,baseFloodElevation,lowestAdjacentGrade,numberOfFloorsInTheInsuredBuilding,obstructionType,originalConstructionDate,...,is_lag_missing,is_elev_diff_missing,is_bfe_missing,median_home_value,imputation,FIPS,STATE,COUNTY_NAME,SHORELINE_FLAG,WATERSHED_FLAG
0,False,False,Unknown,False,0,0,0,2.0,Unknown,1950-07-01,...,1,1,1,1000001.0,NaN,12021,FL,Collier County,1.0,0.0
1,True,False,Unknown,False,0,0,0,1.0,Unknown,1974-01-01,...,1,1,1,79400.0,NaN,29087,MO,Holt County,0.0,0.0
2,True,False,Unknown,False,0,0,0,1.0,Unknown,1978-01-01,...,1,1,1,381700.0,NaN,12021,FL,Collier County,1.0,0.0
3,True,False,None,False,0,0,0,1.0,Unknown,1983-06-15,...,1,1,1,401200.0,NaN,12071,FL,Lee County,0.0,0.0
4,True,False,Finished Basement,False,0,0,0,3.0,Unknown,1975-01-01,...,1,1,1,409600.0,NaN,08013,CO,Boulder County,0.0,0.0


In [11]:
print(f"Records before dropping missing counties: {len(merged_df):,}")

# Drop records where the NOAA county merge did not find a matching county
merged_df = merged_df.dropna(subset=['COUNTY_NAME'])

print(f"Records after dropping missing counties: {len(merged_df):,}")

Records before dropping missing counties: 291,084
Records after dropping missing counties: 290,868


### 9. Data Export & Repository Setup
Creates the standard GitHub repository folder structure (`data/raw/`, `data/processed/`, and `src/`), exports the finalized dataset, and copies raw sources over.

In [12]:
import os
import shutil
import pandas as pd

print("--- Setting up Repository Structure & Exporting Data ---")

# Define the base directory for the GitHub repo
base_dir = '/content/drive/MyDrive/Programming/nfip/Single-Residence Severity'
raw_dir = os.path.join(base_dir, 'data', 'raw')
nhgis_raw_dir = os.path.join(raw_dir, 'nhgis')
processed_dir = os.path.join(base_dir, 'data', 'processed')
src_dir = os.path.join(base_dir, 'src')

# Create directories
os.makedirs(nhgis_raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)
os.makedirs(src_dir, exist_ok=True)

# Fix column types for Parquet compatibility safely
merged_df['median_home_value'] = pd.to_numeric(merged_df['median_home_value'], errors='coerce')
merged_df['imputation'] = merged_df['imputation'].fillna('none').astype(str)

# 1. Save the processed data
parquet_out = os.path.join(processed_dir, 'model_ready_nfip.parquet')
print(f"Saving processed data to {parquet_out}...")
merged_df.to_parquet(parquet_out, index=False)

# 2. Copy the raw files to data/raw for GitHub consistency
raw_nfip_src = '/content/drive/MyDrive/Programming/nfip/FimaNfipClaimsV2.parquet'
raw_nfip_dest = os.path.join(raw_dir, 'FimaNfipClaimsV2.parquet')

if os.path.exists(raw_nfip_src) and not os.path.exists(raw_nfip_dest):
    print("Copying FimaNfipClaimsV2.parquet to data/raw/...")
    shutil.copy2(raw_nfip_src, raw_nfip_dest)

# Copy NHGIS directory contents
src_nhgis_dir = '/content/drive/MyDrive/Programming/nfip/nhgis'
if os.path.exists(src_nhgis_dir):
    print("Copying NHGIS CSVs to data/raw/nhgis/...")
    for file_name in os.listdir(src_nhgis_dir):
        if file_name.endswith('.csv'):
            src_file = os.path.join(src_nhgis_dir, file_name)
            dest_file = os.path.join(nhgis_raw_dir, file_name)
            if not os.path.exists(dest_file):
                shutil.copy2(src_file, dest_file)

print(f"\nSuccess! Repository structure is ready at: {base_dir}")


--- Setting up Repository Structure & Exporting Data ---
Saving processed data to /content/drive/MyDrive/Programming/nfip/Single-Residence Severity/data/processed/model_ready_nfip.parquet...
Copying NHGIS CSVs to data/raw/nhgis/...

Success! Repository structure is ready at: /content/drive/MyDrive/Programming/nfip/Single-Residence Severity
